In [ ]:
import os
import random
import shutil
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from PIL import Image

import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import Dataset,DataLoader
from torchvision import datasets,transforms
from sklearn.metrics import confusion_matrix
from torchvision import transforms

In [ ]:
print(torch.cuda.is_available())
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(device)

In [ ]:
from google.colab import drive
drive.mount("/content/drive")

In [ ]:
import os

DATASET_ROOT = "/content/drive/MyDrive/Datasets"

SOURCE_TRAIN_DIR = os.path.join(DATASET_ROOT, "Training")
CLASSES = ['glioma','meningioma','notumor','pituitary']

SAMPLES_PER_CLASS = 500
TRAIN_PER_CLASS = 100
VAL_PER_CLASS = 50
TEST_PER_CLASS = 50

print("Dataset path :",DATASET_ROOT)
print("Trainig path :",SOURCE_TRAIN_DIR)

# Dataset preparation

In [ ]:
SEED = 42
random.seed(SEED)
torch.manual_seed(SEED)

WORKING_DIR = "/content/BRAIN_TUMOR"

if os.path.exists(WORKING_DIR):
    if os.path.isdir(WORKING_DIR):
        shutil.rmtree(WORKING_DIR)
    elif os.path.isfile(WORKING_DIR):
        os.remove(WORKING_DIR)

for split in ['train','val','test']:
  for cls in CLASSES:
    os.makedirs(os.path.join(WORKING_DIR,split,cls),exist_ok=True)


IMAGE_EXTENSION = ('.jpg','.jepg','.png','.bmp','.webp')

for cls in CLASSES:
  source_cls_dir = os.path.join(SOURCE_TRAIN_DIR,cls)


  if not os.path.isdir(source_cls_dir):
    raise FileNotFoundError(
        f"The expected directory for class '{cls}' was not found: '{source_cls_dir}'. "
        "Please ensure your Google Drive is correctly structured, "
        "or that the 'DATASET_ROOT' and 'SOURCE_TRAIN_DIR' variables "
        "in the previous cell point to the correct location of your dataset."
    )

  image_files =[
      f for f in os.listdir(source_cls_dir)
      if f.lower().endswith(IMAGE_EXTENSION)
  ]

  random.shuffle(image_files)

  if len(image_files) < SAMPLES_PER_CLASS:
    raise ValueError(
        f"{cls} has only {len(image_files)} images,"
        f"but {SAMPLES_PER_CLASS} are required."

    )


  selected = image_files[:SAMPLES_PER_CLASS]
  train_files = selected[:TRAIN_PER_CLASS]
  val_files = selected[TRAIN_PER_CLASS:TRAIN_PER_CLASS + VAL_PER_CLASS]
  test_files = selected[TRAIN_PER_CLASS+VAL_PER_CLASS:]

  splits = {
      "train": train_files,
      "val" : val_files,
      "test" : test_files
  }

  for split,files in splits.items():
    for filename in files:
      src = os.path.join(source_cls_dir,filename)
      dst = os.path.join(WORKING_DIR,split,cls,filename)
      shutil.copy2(src,dst)

print("Dataset preparation completed")


# Data preprocessing

In [ ]:
IMAGE_SIZE = 128
BATCH_SIZE = 32

train_tranforms = transforms.Compose([
    transforms.Resize((IMAGE_SIZE,IMAGE_SIZE)),
    transforms.ToTensor(),
    transforms.Normalize(
        mean=[0.5,0.5,0,5],
        std=[0.5,0.5,0.5]
    )
]
)

eval_tranforms = transforms.Compose([
    transforms.Resize((IMAGE_SIZE,IMAGE_SIZE)),
    transforms.ToTensor(),
    transforms.Normalize(
        mean=[0.5,0.5,0.5],
        std=[0.5,0.5,0.5]
    )
])



In [ ]:

train_tranforms = transforms.Compose([
    transforms.Resize((IMAGE_SIZE,IMAGE_SIZE)),
    transforms.RandomHorizontalFlip(p=0.5),
    transforms.RandomRotation(10),
    transforms.RandomAffine(
        degrees=0,
        translate=(0.5,0.5)
    ),

    transforms.ToTensor(),
    transforms.Normalize(
        mean=[0.5,0.5,0.5],
        std=[0.5,0.5,0.5]
    )
])

eval_tranforms = transforms.Compose([
    transforms.Resize((IMAGE_SIZE,IMAGE_SIZE)),
    transforms.ToTensor(),

    transforms.Normalize(
        mean=[0.5,0.5,0.5],
        std=[0.5,0.5,0.5]
    )
])


In [ ]:
# Crate datasets

train_dataset = datasets.ImageFolder(
    os.path.join(WORKING_DIR,"train"),
    transform = train_tranforms
)

val_dataset = datasets.ImageFolder(
    os.path.join(WORKING_DIR,"val"),
    transform = eval_tranforms
)

test_dataset = datasets.ImageFolder(
    os.path.join(WORKING_DIR,"test"),
    transform =eval_tranforms
)

print("Class name : ",train_dataset.classes)
print("Class to index :",train_dataset.class_to_idx)
print("Train :", len(train_dataset))
print("Validation :",len(val_dataset))
print("Test :",len(test_dataset))

# DataLoader

In [ ]:
train_loader = DataLoader(
    train_dataset,
    batch_size=32,
    shuffle = True,
    num_workers=2
)

test_loader = DataLoader(
    test_dataset,
    batch_size=32,
    num_workers=2
)

val_loader = DataLoader(
    val_dataset,
    batch_size =32,
    num_workers = 2
)

In [ ]:
class SimpleCNN(nn.Module):
  def __init__(self,num_classes =4):
    super().__init__()


    self.features = nn.Sequential(
        nn.Conv2d(3,16, kernel_size=3, padding=1),
        nn.ReLU(),
        nn.MaxPool2d(2),

        nn.Conv2d(16,32,kernel_size=3,padding=1),
        nn.ReLU(),
        nn.MaxPool2d(2),

        nn.Conv2d(32,64,kernel_size=3,padding=1),
        nn.ReLU(),
        nn.MaxPool2d(2)

    )

    self.classifier = nn.Sequential(
        nn.Flatten(),
        nn.Linear(64*16*16,128),
        nn.Dropout(0.3), # Fixed typo: removed extra 'nn,'

        nn.Linear(128,num_classes)
    )

  def forward(self,x):
    x = self.features(x)
    x = self.classifier(x)

    return x

model = SimpleCNN(num_classes=len(CLASSES)) # Pass integer directly
model.to(device) # Move the model to the device after instantiation

print(model)

In [ ]:

epochs = 100
criterion = nn.CrossEntropyLoss()

optimizer = optim.Adam(
    model.parameters(),
    lr=0.001
)

In [ ]:
def train_one_epoch(model,loader,criterion,optimizer,device):
  model.train()

  running_loss = 0.0
  correct = 0
  total = 0

  for images , labels in loader:
    images = images.to(device)
    labels = labels.to(device)

    optimizer.zero_grad()

    outputs = model(images)

    loss = criterion(outputs,labels)
    loss.backward()
    optimizer.step()


    running_loss += loss.item() * images.size(0)

    _,predicted = torch.max(outputs,1)
    correct += (predicted == labels).sum().item()
    total += labels.size(0)
  loss = running_loss / total
  accuracy = correct / total

  return loss,accuracy




In [ ]:
def evaluate(model,loader,criterion,device):
  model.eval()
  running_loss = 0.0
  correct = 0
  total = 0
  with torch.no_grad():
    for images , labels in loader:
     images = images.to(device)
     labels = labels.to(device)

     outputs = model(images)
     loss = criterion(outputs,labels)

     running_loss += loss.item() * images.size(0)
     _,predicted = torch.max(outputs,1)
     correct += (predicted == labels).sum().item()
     total += labels.size(0)
  loss = running_loss / total
  accuracy = correct / total
  return loss,accuracy

In [ ]:
for epoch in range (epochs):
  train_loss,train_acc = train_one_epoch(
      model,
      train_loader,
      criterion,
      optimizer,
      device
  )

  val_loss , val_acc = evaluate(
      model,
      val_loader,
      criterion,
      device
  )



  print(f"Epoch : {epoch + 1}/{epochs}, Train_Loss : {train_loss}, Val_loss : {val_loss}")

In [ ]:
model.eval()

all_predictions = []
all_labels = []

with torch.no_grad():
  for images,labels in test_loader:
    images = images.to(device)


    outputs = model(images)
    predictions = torch.argmax(outputs,dim=1)

    all_predictions.extend(predictions.cpu().numpy())
    all_labels.extend(labels.numpy())

all_predictions =  np.array(all_predictions)

all_labels = np.array(all_labels)

test_accuracy = (all_predictions == all_labels).mean()

print(f"Test Accuracy : {test_accuracy :.4f}")

# Prediction

In [ ]:
def predict_image(model,image_path, class_names,device):
  model.eval()

  image = Image.open(image_path).convert("RGB")

  input_tensor = eval_tranforms(image)
  input_tensor = input_tensor.unsqueeze(0).to(device)

  with torch.no_grad():
    outputs = model(input_tensor)

    probabilities = torch.softmax(outputs,dim=1)
    confidence,predicted_idx = torch.max(probabilities,dim=1)

  predicted_class = class_names[predicted_idx.item()]
  confidence = confidence.item()

  return image,predicted_class,confidence,probabilities.cpu()


# Model Prediction

In [ ]:
sample_path = test_dataset.samples[7][0]
image,predicted_class,confidence,probabilities = predict_image(
    model,
    sample_path,
    test_dataset.classes,
    device
)

plt.figure(figsize=(5,5))
plt.imshow(image)
plt.title(
    f"Prediction :{predicted_class}\n"
    f"Confidence :{confidence:.2%}"
)
plt.axis("off")
plt.show()